<a href="https://colab.research.google.com/github/yd659/Accenture-1K-contract-review-challenge/blob/test/Yutika_Deshpande_BTT_CUAD_Chunking_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import pandas as pd
import json
import re
from collections import defaultdict

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
with open("train_separate_questions.json", "r") as f:
    train_data = json.load(f)

with open("test.json", "r") as f:
    test_data = json.load(f)

category_descriptions = pd.read_csv("category_descriptions.csv")

print("Files loaded successfully.")

FileNotFoundError: [Errno 2] No such file or directory: 'train_separate_questions.json'

In [ ]:
print("train_data type:", type(train_data))
print("test_data type:", type(test_data))

if isinstance(test_data, list):
    print("Number of test records:", len(test_data))
    print("First record type:", type(test_data[0]))

    if isinstance(test_data[0], dict):
        print("First record keys:", test_data[0].keys())

In [ ]:
print("Train contracts:", len(train_data["data"]))
print("Test contracts:", len(test_data["data"]))
print("Total contracts:", len(train_data["data"]) + len(test_data["data"]))

print("\nCategory descriptions shape:")
print(category_descriptions.shape)

In [ ]:
display(category_descriptions)

In [ ]:
print(category_descriptions.columns.tolist())

In [ ]:
contract = train_data["data"][0]
paragraph = contract["paragraphs"][0]

print("Contract title:")
print(contract["title"])

print("\nContract keys:")
print(contract.keys())

print("\nParagraph keys:")
print(paragraph.keys())

print("\nNumber of QAs:")
print(len(paragraph["qas"]))

In [ ]:
for qa in paragraph["qas"][:3]:

    print("ID:", qa["id"])
    print("Question:", qa["question"])
    print("Is impossible:", qa["is_impossible"])

    if qa["answers"]:
        print("Answer:", qa["answers"][0]["text"])
        print("Answer start:", qa["answers"][0]["answer_start"])
    else:
        print("Answer: NONE")

    print("-" * 80)

In [ ]:
def get_category(qa):
    category = qa["id"].split("__")[-1]

    # Remove ending _0, _1, _2, _15, etc.
    category = re.sub(r'_\d+$', '', category)

    return category

print(get_category({
    "id": "SomeContract__Audit Rights_14"
}))

In [ ]:
unique_categories = set()

for dataset in [train_data, test_data]:
    for contract in dataset["data"]:
        for paragraph in contract["paragraphs"]:
            for qa in paragraph["qas"]:
                unique_categories.add(get_category(qa))

unique_categories = sorted(unique_categories)

print("Number of unique categories:", len(unique_categories))

for category in unique_categories:
    print(category)

In [ ]:
def count_positive_contracts(data):

    counts = defaultdict(int)

    for contract in data["data"]:

        categories_in_contract = set()

        for paragraph in contract["paragraphs"]:
            for qa in paragraph["qas"]:

                if len(qa["answers"]) > 0:
                    category = get_category(qa)
                    categories_in_contract.add(category)

        # Count each category at most once per contract
        for category in categories_in_contract:
            counts[category] += 1

    return counts


train_counts = count_positive_contracts(train_data)
test_counts = count_positive_contracts(test_data)

In [ ]:
rows = []

for category in unique_categories:

    train_count = train_counts.get(category, 0)
    test_count = test_counts.get(category, 0)

    rows.append({
        "Category": category,
        "Train Contracts": train_count,
        "Test Contracts": test_count,
        "Total Contracts": train_count + test_count
    })


category_df = pd.DataFrame(rows)

category_df = category_df.sort_values(
    "Train Contracts",
    ascending=False
).reset_index(drop=True)

display(category_df)

In [ ]:
num_train = len(train_data["data"])
num_test = len(test_data["data"])
num_total = num_train + num_test

category_df["Train %"] = (
    category_df["Train Contracts"] / num_train * 100
).round(2)

category_df["Test %"] = (
    category_df["Test Contracts"] / num_test * 100
).round(2)

category_df["Total %"] = (
    category_df["Total Contracts"] / num_total * 100
).round(2)

display(category_df)

In [ ]:
plot_df = category_df.sort_values(
    "Train Contracts",
    ascending=True
)

plt.figure(figsize=(12, 14))

plt.barh(
    plot_df["Category"],
    plot_df["Train Contracts"]
)

plt.xlabel("Number of Training Contracts")
plt.ylabel("Clause Category")
plt.title("CUAD Class Distribution — Positive Contracts per Category")

plt.tight_layout()
plt.show()

In [ ]:
priority_categories = [
    "Cap On Liability",
    "Uncapped Liability",
    "Termination For Convenience",
    "Governing Law",
    "Renewal Term",
    "Ip Ownership Assignment",
    "Change Of Control",
    "Non-Compete",
    "Audit Rights",
    "Most Favored Nation"
]

In [ ]:
for category in priority_categories:

    if category in unique_categories:
        print("FOUND:  ", category)
    else:
        print("MISSING:", category)

In [ ]:
priority_df = category_df[
    category_df["Category"].isin(priority_categories)
].copy()

priority_df = priority_df.sort_values(
    "Train Contracts",
    ascending=False
).reset_index(drop=True)

display(priority_df)

In [ ]:
priority_plot = priority_df.sort_values(
    "Train Contracts",
    ascending=True
)

plt.figure(figsize=(10, 6))

plt.barh(
    priority_plot["Category"],
    priority_plot["Train Contracts"]
)

plt.xlabel("Number of Training Contracts")
plt.ylabel("Priority Category")
plt.title("Training Examples for Priority Clause Categories")

plt.tight_layout()
plt.show()

In [ ]:
imbalance_df = category_df.copy()

imbalance_df["Positive"] = imbalance_df["Train Contracts"]

imbalance_df["Negative"] = (
    num_train - imbalance_df["Train Contracts"]
)

imbalance_df["Positive:Negative Ratio"] = (
    imbalance_df["Positive"] /
    imbalance_df["Negative"].replace(0, 1)
).round(3)

display(
    imbalance_df[
        [
            "Category",
            "Positive",
            "Negative",
            "Positive:Negative Ratio"
        ]
    ]
)

In [ ]:
rarest_categories = category_df.sort_values(
    "Train Contracts",
    ascending=True
)

display(
    rarest_categories[
        ["Category", "Train Contracts", "Train %"]
    ]
)

In [ ]:
contract_lengths = []

for contract in train_data["data"]:

    full_text = " ".join(
        paragraph["context"]
        for paragraph in contract["paragraphs"]
    )

    contract_lengths.append({
        "Contract": contract["title"],
        "Characters": len(full_text),
        "Words": len(full_text.split())
    })


length_df = pd.DataFrame(contract_lengths)

display(length_df.head())

In [ ]:
display(
    length_df[["Characters", "Words"]].describe()
)

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(
    length_df["Words"],
    bins=30
)

plt.xlabel("Approximate Number of Words")
plt.ylabel("Number of Contracts")
plt.title("CUAD Training Contract Length Distribution")

plt.tight_layout()
plt.show()

In [ ]:
def count_answer_spans(data):

    counts = defaultdict(int)

    for contract in data["data"]:
        for paragraph in contract["paragraphs"]:
            for qa in paragraph["qas"]:

                category = get_category(qa)

                counts[category] += len(qa["answers"])

    return counts


train_span_counts = count_answer_spans(train_data)

In [ ]:
category_df["Train Answer Spans"] = (
    category_df["Category"]
    .map(train_span_counts)
    .fillna(0)
    .astype(int)
)

display(
    category_df[
        [
            "Category",
            "Train Contracts",
            "Train Answer Spans",
            "Train %"
        ]
    ]
)

In [ ]:
print("=== EDA SANITY CHECK ===")

print("Train contracts:", len(train_data["data"]))
print("Test contracts:", len(test_data["data"]))
print("Total contracts:",
      len(train_data["data"]) + len(test_data["data"]))

print("Unique categories:", len(unique_categories))

print("\nMost common category:")
display(category_df.head(1))

print("\nRarest category:")
display(category_df.tail(1))

print("\nPriority categories:")
display(priority_df)

# Chunk-Based Multi-Label Data Preparation

This section converts each contract into overlapping 512-token chunks and maps
CUAD's annotated clause spans to the appropriate chunks. Each resulting chunk
will retain its contract ID, clause labels, and supporting evidence.

In [ ]:
!pip install -q transformers

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "distilroberta-base",
    use_fast=True
)

print("Tokenizer loaded successfully.")
print("Maximum model length:", tokenizer.model_max_length)

In [ ]:
def extract_annotations(contract):
    annotations = []

    for paragraph_index, paragraph in enumerate(contract["paragraphs"]):
        for qa in paragraph["qas"]:
            category = get_category(qa)

            for answer in qa["answers"]:
                start = answer["answer_start"]
                text = answer["text"]

                annotations.append({
                    "paragraph_index": paragraph_index,
                    "category": category,
                    "answer_text": text,
                    "answer_start": start,
                    "answer_end": start + len(text)
                })

    return annotations

Tests turning the annotations out of the nested JSON and turning them into a simpler list on one contract

In [ ]:
sample_contract = train_data["data"][0]
sample_annotations = extract_annotations(sample_contract)

print("Contract:", sample_contract["title"])
print("Number of annotated spans:", len(sample_annotations))

sample_annotations[:3]

Verify the annotation positions

Now the 512-token chunks with 64 token overlap need to be created

In [ ]:
def create_chunks(context, max_length=512, overlap=64):
    # Reserve 2 positions for RoBERTa special tokens
    content_length = max_length - 2
    step = content_length - overlap

    encoded = tokenizer(
        context,
        add_special_tokens=False,
        return_offsets_mapping=True,
        truncation=False
    )

    token_ids = encoded["input_ids"]
    offsets = encoded["offset_mapping"]
    chunks = []

    for chunk_id, token_start in enumerate(
        range(0, len(token_ids), step)
    ):
        token_end = min(token_start + content_length, len(token_ids))
        window_offsets = offsets[token_start:token_end]

        chunk_start = window_offsets[0][0]
        chunk_end = window_offsets[-1][1]

        chunks.append({
            "chunk_id": chunk_id,
            "chunk_start": chunk_start,
            "chunk_end": chunk_end,
            "chunk_text": context[chunk_start:chunk_end],
            "token_count": (token_end - token_start) + 2,
            "labels": [],
            "evidence": []
        })

        if token_end == len(token_ids):
            break

    return chunks

In [ ]:
sample_context = sample_contract["paragraphs"][0]["context"]
sample_chunks = create_chunks(sample_context)

print("Contract characters:", len(sample_context))
print("Chunks:", len(sample_chunks))
print("First range:",
      sample_chunks[0]["chunk_start"],
      sample_chunks[0]["chunk_end"])
print("Last range:",
      sample_chunks[-1]["chunk_start"],
      sample_chunks[-1]["chunk_end"])
print("Maximum tokens:",
      max(chunk["token_count"] for chunk in sample_chunks))

Test the function, which should break the contract into inputs of no more than 512 tokens and repeats about 64 tokens between neighboring chunks. It also creates empty labels and evidence fields for the next step.

In [ ]:
sample_context = sample_contract["paragraphs"][0]["context"]
sample_chunks = create_chunks(sample_context)

print("Contract characters:", len(sample_context))
print("Number of chunks:", len(sample_chunks))
print("Maximum token count:", max(
    chunk["token_count"] for chunk in sample_chunks
))

Inspects the first chunk

In [ ]:
first_chunk = sample_chunks[0]

print("Chunk ID:", first_chunk["chunk_id"])
print("Character range:",
      first_chunk["chunk_start"],
      "to",
      first_chunk["chunk_end"])
print("Token count:", first_chunk["token_count"])
print("\nChunk preview:\n")
print(first_chunk["chunk_text"][:1000])

Inspects the start and end of the first 2 chunks

In [ ]:
for chunk in sample_chunks[:2]:
    print("=" * 80)
    print("Chunk ID:", chunk["chunk_id"])
    print("Range:", chunk["chunk_start"], "-", chunk["chunk_end"])
    print("Token count:", chunk["token_count"])
    print("Beginning:", repr(chunk["chunk_text"][:100]))
    print("End:", repr(chunk["chunk_text"][-100:]))

Checks the size

In [ ]:
oversized_chunks = [
    chunk for chunk in sample_chunks
    if chunk["token_count"] > 512
]

print("Oversized chunks:", len(oversized_chunks))

Now need to add annotations to chunks

In [ ]:
def label_chunks(chunks, annotations):
    unassigned = []

    for annotation in annotations:
        answer_start = annotation["answer_start"]
        answer_end = annotation["answer_end"]
        category = annotation["category"]

        fully_contained_chunks = []
        overlapping_chunks = []

        for chunk in chunks:
            chunk_start = chunk["chunk_start"]
            chunk_end = chunk["chunk_end"]

            # Annotation is completely inside the chunk
            if answer_start >= chunk_start and answer_end <= chunk_end:
                fully_contained_chunks.append(chunk)

            # Number of characters shared by annotation and chunk
            overlap = max(
                0,
                min(answer_end, chunk_end)
                - max(answer_start, chunk_start)
            )

            if overlap > 0:
                overlapping_chunks.append((overlap, chunk))

        # Prefer chunks containing the complete annotation
        if fully_contained_chunks:
            target_chunks = fully_contained_chunks
            assignment_type = "fully_contained"

        # Otherwise use the chunk with the greatest overlap
        elif overlapping_chunks:
            best_overlap, best_chunk = max(
                overlapping_chunks,
                key=lambda item: item[0]
            )
            target_chunks = [best_chunk]
            assignment_type = "maximum_overlap"

        else:
            unassigned.append(annotation)
            continue

        for chunk in target_chunks:
            if category not in chunk["labels"]:
                chunk["labels"].append(category)

            chunk["evidence"].append({
                **annotation,
                "assignment_type": assignment_type
            })

    return chunks, unassigned

Runs the above on a sample contract

In [ ]:
labeled_chunks, unassigned_annotations = label_chunks(
    sample_chunks,
    sample_annotations
)

print("Total chunks:", len(labeled_chunks))
print(
    "Labeled chunks:",
    sum(len(chunk["labels"]) > 0 for chunk in labeled_chunks)
)
print(
    "Unlabeled chunks:",
    sum(len(chunk["labels"]) == 0 for chunk in labeled_chunks)
)
print("Unassigned annotations:", len(unassigned_annotations))

Now, process all contracts

In [ ]:
def process_contracts(contracts, source):
    rows = []
    unassigned_records = []

    for contract_index, contract in enumerate(contracts):
        contract_id = contract.get(
            "title",
            f"{source}_contract_{contract_index}"
        )

        context = contract["paragraphs"][0]["context"]
        annotations = extract_annotations(contract)

        chunks = create_chunks(context)
        labeled_chunks, unassigned = label_chunks(
            chunks,
            annotations
        )

        for chunk in labeled_chunks:
            rows.append({
                "contract_id": contract_id,
                "source": source,
                "chunk_id": chunk["chunk_id"],
                "chunk_start": chunk["chunk_start"],
                "chunk_end": chunk["chunk_end"],
                "chunk_text": chunk["chunk_text"],
                "token_count": chunk["token_count"],
                "labels": chunk["labels"],
                "evidence": chunk["evidence"]
            })

        for annotation in unassigned:
            unassigned_records.append({
                "contract_id": contract_id,
                **annotation
            })

    return rows, unassigned_records

In [ ]:
train_rows, train_unassigned = process_contracts(
    train_data["data"],
    source="original_train"
)

test_rows, test_unassigned = process_contracts(
    test_data["data"],
    source="original_test"
)

In [ ]:
all_rows = train_rows + test_rows
chunked_df = pd.DataFrame(all_rows)

In [ ]:
print("Total contracts:", chunked_df["contract_id"].nunique())
print(
    chunked_df.groupby("source")["contract_id"]
    .nunique()
)
print("Total chunks:", len(chunked_df))
print("Maximum tokens:", chunked_df["token_count"].max())
print("Train unassigned:", len(train_unassigned))
print("Test unassigned:", len(test_unassigned))

In [ ]:
chunked_df.to_json(
    "cuad_all_510_chunked_multilabel.jsonl",
    orient="records",
    lines=True
)

print("Combined 510-contract dataset saved.")

In [ ]:
print(type(test_data))

if isinstance(test_data, dict):
    test_contracts = test_data["data"]
else:
    test_contracts = test_data

print("Test contracts:", len(test_contracts))
print("First contract keys:", test_contracts[0].keys())

Process the training data - test on 5 contracts first

Check the label distribution

In [ ]:
label_counts = (
    chunked_df.explode("labels")["labels"]
    .dropna()
    .value_counts()
)

display(label_counts)

Check how many chunks have no clause label

In [ ]:
chunked_df["has_label"] = chunked_df["labels"].apply(
    lambda labels: len(labels) > 0
)

print(chunked_df["has_label"].value_counts())

Save the result

In [ ]:
chunked_df.to_json(
    "cuad_chunked_multilabel.jsonl",
    orient="records",
    lines=True
)

Download the file

In [ ]:
from google.colab import files

files.download("cuad_chunked_multilabel.jsonl")

Checking the json file

In [ ]:
chunked_df = pd.read_json(
    "cuad_chunked_multilabel.jsonl",
    lines=True
)

print("Rows:", len(chunked_df))
print("Contracts:", chunked_df["contract_id"].nunique())
print("Maximum tokens:", chunked_df["token_count"].max())
print("Missing contract IDs:", chunked_df["contract_id"].isna().sum())
print("Missing text:", chunked_df["chunk_text"].isna().sum())
print(
    "Labeled chunks:",
    chunked_df["labels"].apply(lambda x: len(x) > 0).sum()
)

In [ ]:
print("Globally unassigned annotations:", len(train_unassigned))